In [12]:
from scipy.special import comb
import os
import pandas as pd
from statsmodels.stats.weightstats import ttest_ind
import numpy as np
from scipy.stats import mannwhitneyu
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
from scipy.stats import expon

import os
if 'COLAB_GPU' in os.environ:
  from google.colab import drive
  drive.mount("/content/drive/")

  path = "/content/drive/MyDrive/Estudios/ding/data"
  df = pd.read_csv(os.path.join(path, "Penn46_ascii.txt"), delimiter=" ")

else:
  df = pd.read_csv("data/Penn46_ascii.txt", delimiter = " ")

# Estratificación y Post-estratificación en Experimentos Aleatorizados

## Estratificación

<br>

En un experimento completamente aleatorizado, donde las unidades tienen una covariable discreta con $k$ estratos, es posible que la asignación de tratamiento y control genere un desbalance de dicha covariable, causando que sea dificil de interpretar los resultados del experimento ya que estos podrían ser atribuidos por el desbalance de la caovariable o por el tratamiento.  


Para evitar esto, es posible utilizar u experimento aleatorizado estratificado (SRE).

<br>

**Definición: Experimento aleatorizado estratificado**

Considera un CRE con $n$ unidades. Adicionalmente, considera una covariable discreta $X_i \in \{ i,...,K\}$ donde podemos definir el número y proporción de unidades de cada estrato como  $n_{[k]} = \#\{i:X_i=k\}$ y $\pi_{[k]}=n_{[k]}/n$, respectivamente.

El número de unidades del estrato $k$ asignadas al tratamiento ($n_{[k]1}$) y al control ($n_{[k]0}$) es:

$$n_{[k]1}=\#\{i:X_i=k, Z_i = 1\}$$
$$n_{[k]0}=\#\{i:X_i=k, Z_i = 0\}$$

<br>

Consideremos estas unidades como fijas. Un SRE es un experimento en el que se conducen $K$ CRE independientes dentro de los K estratos de la covariable discreta $X$.

<br>

El número total de aleatorizaciones posibles es:

$$\prod_{i=1}^k\binom{n_{[k]}}{n_{[k]1}} < \binom{n}{n_1}$$

<br>

**Definición: Propensity score**

El propensity score es la proporción de unidades de cada estrato que reciben el tratamiento:

$$e_{[k]}=\frac{n_{[k]1}}{n_{[k]}}$$

- $e_{[k]}$ es fijo en un SRE pero aleatorio en un CRE.

**Definición: Efectos causales promedio del estrato y efecto causal promedio global**

recordemos que el efecto causal para cada unidad $i$ es $\tau_i = Y_i(1) - Y_i(0)$. Para cada estrato $k$, podemos definir el efecto causal promedio:

$$\tau_{[k]} = n_{[k]}^{-1}\sum_{X_i=k}\tau_i$$

El efecto causal promedio global es el promedio ponderado de los efectos causales promedios de cada estrato, donde el ponderador es la proporción de unidades del estrato.

$$\tau=\sum_{k=1}^K\pi_{[k]}\tau_{[k]}$$

## Prueba de aleatorización de Fisher en Experimentos Estratificados Aleatorizados

Recordemos que la hipotesis nula de Fisher es que los efectos potenciales de la unidad $i$ son iguales:

$$H_{0F}:Y_i(1) =Y_i(0) \text{ para todas las unidades i = 1,...,n}$$

<br>

En el caso estratificado, podemos usar cualquier estadístico $T=T(Z,Y,X)$ donde $Z$ es el vector de tratamiento, $Y$ es el vector de resultados observados y $X$ es la matriz de covariadas. Bajo un SRE y $H_{0F}$ el estadístico T tiene una distribución conocida porque la distribución de $Z$ es conocida. En este caso, debemos obtener la distribución del vector de tratamiento a través de permutaciones en cada estrato, es decir, obtener la distribución condicional de $Z | X_i = k$. A esta prueba se le conoce como prueba de aleatorización condicional o prueba de permutación condicional.

<br>

**Estimador estratificado**

Cuando estratificamos, es posible obtener el  el estimador estratificado del efecto causal promedio $\hat\tau_S$:

$$\hat\tau_S = \sum_{k=1}^K\pi_{[k]}\hat\tau_{[k]}$$

Donde $\hat\tau_{[k]}$ es el estimador del efecto causal promedio del estato $k$:

$$\hat\tau_{[k]} = n_{[k]1}^{-1}\sum_{i=1}^{n}I(Z_i=1, X_i=k)Y_i - n_{[k]0}^{-1}\sum_{i=1}^{n}I(Z_i=0, X_i=k)Y_i$$

<br>

**Estimador estratificado studentizado**

La versión studentizada del estimador del estimador del efecto causal promedio estratificado está dado por:

$$t_S=\frac{\hat\tau_S}{\sqrt{\hat V_S}}$$

Donde $\hat V_S$ es el estimador estratificado de la varianza, asumiendo que no hay correlación entre los estratos:

$$\hat V_S= \sum_{k=1}^K\pi_{[k]}^2\bigg(\frac{\hat S_{[k]}^2(1)}{n_{[k]1}}  + \frac{\hat S_{[k]}^2(0)}{n_{[k]0}}\bigg)$$


y en donde $S_{[k]}^2(1) \text{ y }S_{[k]}^2(0)$ son las varianzas muestrales de los resultados bajo tratamiento y control, respectivamente.

<br>

**Prueba de Wilcoxon para suma de rangos**

Para la prueba de Wilcoxon es posible calcular un valor estratificado $W_S$ mediante la suma ponderada de los estadísticos en cada estrato $W_{[k]}$:

$$W_S = \sum_{k=1}^Kc_{[k]}W_{[k]}$$

Aquí no se discute el cálculo del ponderador, sin embargo, pueden usarse dos:

 - $c_{[k]}=\frac{1}{n_{[k]1}n_{[k]0}}$

 - $c_{[k]}=\frac{1}{n_{[k]}+1}$

<br>

**Prueba de Kolmogorov-Smirnoff**

Podemos computar el estadístico estratificado $D_S$ como la suma ponderada de las diferencias máximas de las distribuciones empíricas de los resultados bajo tratamiento y control en cada estrato $D_{[k]}$:

$$D_S=\sum_{k=1}^Kc_{[k]}D_{[k]}$$

Donde:

$$c_{[k]}= \sqrt{n_{[k]1}n_{[k]0} / n_{[k]}}$$



In [13]:
# Penn data

z = df["treatment"]
y = np.log(df["duration"])
block = df["quarter"]

# Creamos una tabla de contingencia
pd.crosstab(z, block)

quarter,0,1,2,3,4,5
treatment,,,,,,
0,234,41,687,794,738,860
1,87,48,757,866,811,461


In [14]:
def stat_sre(z, y, x):
  xlevels = np.unique(x)
  K = len(xlevels)
  Pi_K = np.zeros(K)
  Tau_K = np.zeros(K)
  W_K = np.zeros(K)

  for k in range(K):
    x_k = xlevels[k]
    z_k = z[x==x_k]
    y_k = y[x==x_k]
    Pi_K[k] = len(z_k)/len(z)
    Tau_K[k] = np.mean(y_k[z_k==1]) - np.mean(y_k[z_k==0])
    W_K[k] = mannwhitneyu(y_k[z_k==1], y_k[z_k==0]).statistic

  return [np.sum(Pi_K * Tau_K), np.sum(W_K / Pi_K)]


In [15]:
stats_obs = stat_sre(z, y, block)
stats_obs

[np.float64(-0.0899064590011073), np.float64(4687961.229438581)]

In [16]:
def zRandomSRE(z, x):
  xlevels = np.unique(x)
  K = len(xlevels)
  zrandom = z.copy()
  for k in range(K):
    x_k = xlevels[k]
    zrandom[x==x_k] = np.random.permutation(z[x==x_k])

  return zrandom

In [20]:
MC = 1000
statSREMC = np.zeros((MC, 2))
print(statSREMC.shape)
for mc in range(MC):
  zrandom = zRandomSRE(z, block)
  statSREMC[mc, ] = stat_sre(zrandom, y, block)

(1000, 2)


In [21]:
print(np.mean(statSREMC[:,0] <= stats_obs[0]))
print(np.mean(statSREMC[:,1] <= stats_obs[1]))

0.001
0.0


## Inferencia Neymaniana



In [19]:
##